# Chapter 3 — Introduction to Neural Prediction: Forward Propagation

**Mata Kuliah:** Deep Learning
**Buku:** Grokking Deep Learning (Andrew W. Trask, Manning, 2019)
**Chapter 3:** Introduction to Neural Prediction: Forward Propagation

Notebook ini mereproduksi kode Chapter 3 dari buku, mengikuti urutan
pembahasan buku. Setiap bagian diawali penjelasan yang disusun dari teks buku.

**Halaman buku:** 21–46

## Isi Chapter 3

1. A simple neural network making a prediction
2. What is a neural network, and what does it do?
3. Making a prediction with multiple inputs
4. Making a prediction with multiple outputs
5. Predicting with multiple inputs and outputs
6. Predicting on predictions (jaringan bertumpuk)
7. A quick primer on NumPy

## Inti Chapter: Forward Propagation

Chapter ini membahas langkah **predict** dari siklus predict, compare, learn.
Neural network membuat prediksi dengan **weighted sum** (jumlah berbobot) dari input.
Prediksi dibuat dengan mempropagasikan aktivasi maju melalui jaringan. Karena itu
disebut **forward propagation**. Aktivasi adalah semua angka selain weights, yang
berbeda untuk setiap prediksi.

Contoh sepanjang chapter: memprediksi peluang tim baseball menang dengan data
pada pertandingan pertama musim:
- `toes`: rata-rata jumlah jari kaki pemain
- `wlrec`: persentase kemenangan saat ini
- `nfans`: jumlah penggemar (dalam juta)

## 1. A Simple Neural Network Making a Prediction

Jaringan paling sederhana: satu input, satu weight, satu prediksi.
Weight adalah **tombol volume**: weight besar memperkuat input, weight kecil
memperlemahnya.

In [1]:
weight = 0.1

def neural_network(input, weight):
    prediction = input * weight
    return prediction

number_of_toes = [8.5, 9.5, 10, 9]
input = number_of_toes[0]

pred = neural_network(input, weight)
print(pred)

0.8500000000000001


Hasilnya `0.85` (8.5 × 0.1). Jaringan tidak mengingat prediksi sebelumnya.
Ia hanya mengetahui input yang diberikan saat itu.

## 2. What Is a Neural Network, and What Does It Do?

Neural network sederhana adalah satu atau lebih weight yang dikalikan dengan
data input untuk membuat prediksi.

- **Input data**: angka yang direkam dari dunia nyata.
- **Prediction**: apa yang dikatakan jaringan berdasarkan input.
- **Belajar**: jika prediksi terlalu tinggi, weight diturunkan; jika terlalu
  rendah, weight dinaikkan (*trial and error*).

Antarmuka neural network selalu sama: menerima `input` sebagai informasi dan
`weight` sebagai pengetahuan, lalu menghasilkan `prediction`.

## 3. Making a Prediction with Multiple Inputs

Jaringan dapat menerima beberapa data sekaligus. Setiap input dikalikan weight-nya
sendiri, lalu hasilnya **dijumlahkan**. Ini disebut **weighted sum** atau
**dot product**.

Input dan weights berbentuk **vektor** (daftar angka). Operasi yang memasangkan
angka berdasarkan posisi disebut **elementwise**.

Intuisi dot product: ukuran **kemiripan** dua vektor. Buku menjelaskannya seperti
logika: weight positif berperilaku seperti AND/OR, dan weight negatif seperti NOT.

### Kode lengkap (Python murni)

Seperti di buku, kode ini memakai list dan angka biasa. Setiap input dikalikan
weight pada posisi yang sama, lalu hasilnya dijumlahkan.

In [2]:
def w_sum(a, b):
    assert(len(a) == len(b))
    output = 0
    for i in range(len(a)):
        output += (a[i] * b[i])
    return output

weights = [0.1, 0.2, 0]

def neural_network(input, weights):
    pred = w_sum(input, weights)
    return pred

toes  = [8.5, 9.5, 9.9, 9.0]
wlrec = [0.65, 0.8, 0.8, 0.9]
nfans = [1.2, 1.3, 0.5, 1.0]

# Input untuk pertandingan pertama musim
input = [toes[0], wlrec[0], nfans[0]]

pred = neural_network(input, weights)
print(pred)

0.9800000000000001


Hasil: `0.98` (0,85 + 0,13 + 0,00). Weight untuk `nfans` bernilai 0, sehingga
jumlah penggemar tidak memengaruhi prediksi.

### Multiple inputs: apa yang dilakukan jaringan ini?

Jaringan ini mengalikan tiga input dengan tiga weight, lalu menjumlahkannya.
Setiap input tetap memiliki "tombol volume"-nya sendiri. Sifat baru hanya
penjumlahan hasil-hasil tersebut.

Hal-hal penting yang disebut buku:
- Weights tidak boleh diacak urutannya, karena tiap posisi punya arti.
- Nilai weight dan nilai input sama-sama menentukan pengaruhnya.
- Weight negatif membuat input tertentu mengurangi prediksi.

## 4. Making a Prediction with Multiple Outputs

Satu input dapat menghasilkan beberapa prediksi sekaligus. Setiap prediksi
memiliki weight sendiri, dan ketiganya **tidak saling terhubung**.

Contoh: dari `win/loss record` (wlrec) saja, jaringan memprediksi tiga hal:
apakah pemain *hurt* (terluka), *win* (menang), dan *sad* (sedih).

In [3]:
def ele_mul(number, vector):
    output = [0, 0, 0]
    assert(len(output) == len(vector))
    for i in range(len(vector)):
        output[i] = number * vector[i]
    return output

weights = [0.3, 0.2, 0.9]

def neural_network(input, weights):
    pred = ele_mul(input, weights)
    return pred

wlrec = [0.65, 0.8, 0.8, 0.9]
input = wlrec[0]

pred = neural_network(input, weights)
print(pred)

[0.195, 0.13, 0.5850000000000001]


Hasil buku: `[0.195, 0.13, 0.585]` (hurt, win, sad). Ketiga prediksi berdiri sendiri
dan masing-masing menerima input yang sama.

## 5. Predicting with Multiple Inputs and Outputs

Gabungan keduanya: banyak input dan banyak output. Setiap output dihitung
sebagai **weighted sum** dari seluruh input dengan satu baris weights.
Weights sekarang berbentuk **matriks** (daftar vektor).

Operasi "vektor dikalikan matriks" dilakukan dengan mengulang weighted sum untuk
setiap baris matriks.

In [4]:
def w_sum(a, b):
    assert(len(a) == len(b))
    output = 0
    for i in range(len(a)):
        output += (a[i] * b[i])
    return output

def vect_mat_mul(vect, matrix):
    assert(len(vect) == len(matrix))
    output = [0, 0, 0]
    for i in range(len(vect)):
        output[i] = w_sum(vect, matrix[i])
    return output

# Baris pertama: hurt?, baris kedua: win?, baris ketiga: sad?
weights = [[0.1, 0.1, -0.3],
           [0.1, 0.2, 0.0],
           [0.0, 1.3, 0.1]]

def neural_network(input, weights):
    pred = vect_mat_mul(input, weights)
    return pred

toes  = [8.5, 9.5, 9.9, 9.0]
wlrec = [0.65, 0.8, 0.8, 0.9]
nfans = [1.2, 1.3, 0.5, 1.0]

input = [toes[0], wlrec[0], nfans[0]]
pred = neural_network(input, weights)
print(pred)

[0.555, 0.9800000000000001, 0.9650000000000001]


Hasil buku: `[0.555, 0.98, 0.965]` (hurt, win, sad).

### Multiple inputs and outputs: cara kerjanya

Jaringan ini melakukan tiga weighted sum independen. Setiap output node memiliki
weighted sum sendiri dari input. Buku menyebut `vect_mat_mul` sebagai
**vector-matrix multiplication**: vektor input dikalikan dengan setiap baris matriks.

## 6. Predicting on Predictions (Jaringan Bertumpuk)

Output satu jaringan dapat menjadi input jaringan berikutnya. Hasilnya dua
perkalian vektor-matriki berurutan. Layer di antaranya disebut **hidden layer**.
Buku menjelaskan bahwa pola yang rumit (misalnya klasifikasi gambar) tidak bisa
ditangkap oleh satu matriks weights saja.

In [5]:
# ih_wgt: dari input ke hidden layer; hp_wgt: dari hidden layer ke prediksi
ih_wgt = [[0.1, 0.2, -0.1],
          [-0.1, 0.1, 0.9],
          [0.1, 0.4, 0.1]]

hp_wgt = [[0.3, 1.1, -0.3],
          [0.1, 0.2, 0.0],
          [0.0, 1.3, 0.1]]

weights = [ih_wgt, hp_wgt]

def neural_network(input, weights):
    hid = vect_mat_mul(input, weights[0])
    pred = vect_mat_mul(hid, weights[1])
    return pred

toes  = [8.5, 9.5, 9.9, 9.0]
wlrec = [0.65, 0.8, 0.8, 0.9]
nfans = [1.2, 1.3, 0.5, 1.0]

input = [toes[0], wlrec[0], nfans[0]]
pred = neural_network(input, weights)
print(pred)

[0.21350000000000002, 0.14500000000000002, 0.5065]


Hasil: `[0.214, 0.145, 0.507]` (hurt, win, sad). Hidden layer menghasilkan
`[0.86, 0.295, 1.23]` sebelum diteruskan ke layer prediksi.

## 7. NumPy Version

NumPy menyediakan fungsi `dot` (dot product) dan operasi matriks. Kode di bawah
menghasilkan hasil yang sama dengan versi Python murni di atas. Ini adalah bentuk
yang akan dipakai pada chapter-chapter berikutnya.

In [6]:
import numpy as np

# Weights sebagai array NumPy (.T untuk memutar matriks sesuai urutan perkalian)
ih_wgt = np.array([
    [0.1, 0.2, -0.1],   # hid[0]
    [-0.1, 0.1, 0.9],   # hid[1]
    [0.1, 0.4, 0.1]]).T # hid[2]

hp_wgt = np.array([
    [0.3, 1.1, -0.3],   # hurt?
    [0.1, 0.2, 0.0],    # win?
    [0.0, 1.3, 0.1]]).T # sad?

weights = [ih_wgt, hp_wgt]

def neural_network(input, weights):
    hid = input.dot(weights[0])
    pred = hid.dot(weights[1])
    return pred

toes  = np.array([8.5, 9.5, 9.9, 9.0])
wlrec = np.array([0.65, 0.8, 0.8, 0.9])
nfans = np.array([1.2, 1.3, 0.5, 1.0])

input = np.array([toes[0], wlrec[0], nfans[0]])
pred = neural_network(input, weights)
print(pred)

[0.2135 0.145  0.5065]


## 8. A Quick Primer on NumPy

Vektor dan matriks dibuat dengan `np.array`. Perhatikan:
- Vektor dengan satu baris dan matriks dengan satu baris sama-sama dibuat dengan cara yang sama.
- Matriks dibuat dengan urutan `(baris, kolom)`.
- Operasi `*`, `+`, `-`, `/` bersifat **elementwise**. Dua operan harus punya jumlah
  kolom yang sama, atau salah satunya hanya satu kolom (skalar).
- Untuk perkalian matriks gunakan `.dot`. Aturannya: kolom kiri harus sama
  dengan baris kanan. `(a, b).dot(b, c) = (a, c)`.

In [7]:
import numpy as np

a = np.array([0, 1, 2, 3])          # vektor
b = np.array([4, 5, 6, 7])          # vektor lain
c = np.array([[0, 1, 2, 3],         # matriks 2x4
              [4, 5, 6, 7]])

d = np.zeros((2, 4))                # matriks 2x4 berisi nol
e = np.random.rand(2, 5)            # matriks 2x5 berisi angka 0-1

print(a)
print(b)
print(c)
print(d)
print(e)

[0 1 2 3]
[4 5 6 7]
[[0 1 2 3]
 [4 5 6 7]]
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]]
[[0.92487809 0.98409074 0.40465508 0.19798658 0.50046326]
 [0.29674195 0.62547221 0.74015116 0.37815848 0.73494225]]


In [8]:
print(a * 0.1)     # setiap angka pada vektor a dikali 0,1
print(c * 0.2)     # setiap angka pada matriks c dikali 0,2
print(a * b)       # elementwise: posisi dipasangkan
print(a * b * 0.2)
print(a * c)       # a dikalikan elementwise dengan setiap baris c

try:
    print(a * e)   # a dan e tidak punya jumlah kolom yang sama
except ValueError as err:
    print('ValueError:', err)

[0.  0.1 0.2 0.3]
[[0.  0.2 0.4 0.6]
 [0.8 1.  1.2 1.4]]
[ 0  5 12 21]
[0.  1.  2.4 4.2]
[[ 0  1  4  9]
 [ 0  5 12 21]]
ValueError: operands could not be broadcast together with shapes (4,) (2,5) 


In [9]:
a = np.zeros((1, 4))
b = np.zeros((4, 3))
c = a.dot(b)
print(c.shape)      # (1, 3)

a = np.zeros((2, 4))
b = np.zeros((4, 3))
c = a.dot(b)
print(c.shape)      # (2, 3)

e = np.zeros((2, 1))
f = np.zeros((1, 3))
g = e.dot(f)
print(g.shape)      # (2, 3)

h = np.zeros((5, 4)).T   # .T memutar baris dan kolom
i = np.zeros((5, 6))
j = h.dot(i)
print(j.shape)      # (4, 6)

h = np.zeros((5, 4))
i = np.zeros((5, 6))
try:
    j = h.dot(i)
except ValueError as err:
    print('ValueError:', err)

(1, 3)
(2, 3)
(2, 3)
(4, 6)
ValueError: shapes (5,4) and (5,6) not aligned: 4 (dim 1) != 5 (dim 0)


## Key Ideas — Chapter 3

- Neural network membuat prediksi dengan **weighted sum** dari input.
- Weight berperan sebagai "tombol volume" yang menentukan sensitivitas prediksi terhadap input.
- **Vektor** adalah daftar angka; **matriks** adalah daftar vektor.
- Banyak input dan banyak output: setiap output memiliki weighted sum sendiri.
- **Jaringan bertumpuk** (hidden layer) membuat prediksi dari prediksi.
- **Forward propagation** = mempropagasikan aktivasi maju untuk menghasilkan prediksi.

## Kesimpulan

Chapter 3 mengajarkan langkah **predict**. Semua jaringan di chapter ini
dibangun dari satu aturan yang diulang: weighted sum. Kecerdasan jaringan berada
pada nilai weights. Chapter 4 membahas cara menentukan weights tersebut melalui
**compare** dan **learn**, yaitu gradient descent.